# Spaceship Titanic — End-to-end ML pipeline

This notebook contains the full end-to-end pipeline : data loading, EDA, feature engineering, preprocessing, a scikit-learn pipeline with a RandomForest, quick evaluation, and saving artifacts.

Some heavy hyperparameter search cells are included but commented or limited to avoid excessive compute when run unintentionally.

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
import joblib
import matplotlib.pyplot as plt

# Path to training CSV that was uploaded in the session
DATA_PATH = "train.csv"
print('Expecting train CSV at:', DATA_PATH)
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Data file not found at {DATA_PATH} — upload it or change DATA_PATH")

df = pd.read_csv(DATA_PATH)
print('Loaded dataset shape:', df.shape)


In [ ]:
# Visualize relationship between numerical features and target (using boxplots)
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(15, 15))
axes = axes.flatten()

# Preprocessing and pipeline
fe = FeatureEngineer()
X_fe = fe.transform(X)

# Create a temporary DataFrame for plotting with the target variable
df_plot = pd.concat([X_fe[numeric_cols], y], axis=1)

for i, col in enumerate(numeric_cols):
    if col in df_plot.columns and target_col in df_plot.columns:
        df_plot.boxplot(column=col, by=target_col, ax=axes[i])
        axes[i].set_title(f'{col} vs {target_col}')
        axes[i].set_xlabel(target_col)
        axes[i].set_ylabel(col)
        plt.suptitle('') # Hide the default suptitle
    else:
        axes[i].set_title(f'{col} or {target_col} not in DataFrame')
        axes[i].axis('off') # Hide the subplot if column not found

plt.tight_layout()
plt.show()

In [ ]:
# Visualize distribution of numerical features
numeric_cols = ['Age', 'cabin_num', 'group_size', 'total_spend', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(15, 15))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    if col in X_fe.columns:
        X_fe[col].hist(ax=axes[i], bins=30)
        axes[i].set_title(f'Distribution of {col}')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Frequency')
    else:
        axes[i].set_title(f'{col} not in DataFrame')
        axes[i].axis('off') # Hide the subplot if column not found

plt.tight_layout()
plt.show()

In [ ]:
# Visualize relationship between categorical features and target
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    if col in X_fe.columns and target_col in df.columns:
        # Create a cross-tabulation of the feature and target
        ct = pd.crosstab(X_fe[col], y)
        ct.plot(kind='bar', stacked=True, ax=axes[i])
        axes[i].set_title(f'{col} vs {target_col}')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Count')
        axes[i].legend(title=target_col, labels=['Not Transported', 'Transported'])
    else:
        axes[i].set_title(f'{col} or {target_col} not in DataFrame')
        axes[i].axis('off') # Hide the subplot if column not found

plt.tight_layout()
plt.show()

In [ ]:
# Visualize distribution of categorical features
categorical_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'deck', 'side']

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    if col in X_fe.columns:
        X_fe[col].value_counts().plot(kind='bar', ax=axes[i])
        axes[i].set_title(f'Distribution of {col}')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Count')
    else:
        axes[i].set_title(f'{col} not in DataFrame')
        axes[i].axis('off') # Hide the subplot if column not found

plt.tight_layout()
plt.show()

## Exploratory Data Analysis (EDA) - Visualizations

Let's visualize the distribution of some key features and their relationship with the target variable `Transported`.

In [ ]:
# Quick preview
display(df.head())
display(df.info())
display(df.isna().sum().sort_values(ascending=False))


In [ ]:
# Target
target_col = 'Transported'
if target_col in df.columns:
    y = df[target_col].map({True:1, False:0})
    X = df.drop(columns=[target_col])
else:
    raise ValueError("Expected target column 'Transported' in uploaded CSV.")

import matplotlib.pyplot as plt
plt.figure()
y.value_counts(normalize=True).plot(kind='bar')
plt.title('Transported distribution (proportion)')
plt.show()


In [ ]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Transformations:
     - parse Cabin -> deck, cabin_num, side
     - extract group id from PassengerId and group size/is_alone
     - numeric-cast service columns and sum to total_spend
     - cast Age, CryoSleep, VIP to numeric flags
    """
    def __init__(self):
        self.service_cols = ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        if 'PassengerId' in X.columns:
            X['PassengerId'] = X['PassengerId'].astype(str)
            X['group'] = X['PassengerId'].str.split('_').str[0]
            X['group_size'] = X.groupby('group')['PassengerId'].transform('count')
            X['is_alone'] = (X['group_size'] == 1).astype(int)
        else:
            X['group_size'] = 1
            X['is_alone'] = 1
        if 'Cabin' in X.columns:
            cabin = X['Cabin'].fillna('Unknown/0/Unknown').astype(str)
            parts = cabin.str.split('/', expand=True)
            X['deck'] = parts[0].replace('Unknown', np.nan)
            X['cabin_num'] = pd.to_numeric(parts[1], errors='coerce')
            X['side'] = parts[2].replace('Unknown', np.nan)
        else:
            X['deck'] = np.nan
            X['cabin_num'] = np.nan
            X['side'] = np.nan
        for col in self.service_cols:
            if col in X.columns:
                X[col] = pd.to_numeric(X[col], errors='coerce').fillna(0.0)
            else:
                X[col] = 0.0
        X['total_spend'] = X[self.service_cols].sum(axis=1)
        if 'Age' in X.columns:
            X['Age'] = pd.to_numeric(X['Age'], errors='coerce')
        else:
            X['Age'] = np.nan
        for col in ['CryoSleep','VIP']:
            if col in X.columns:
                X[col] = X[col].map({True:1, False:0})
            else:
                X[col] = np.nan
        return X


In [ ]:
# Preprocessing and pipeline
fe = FeatureEngineer()
X_fe = fe.transform(X)

numeric_features = ['Age','cabin_num','group_size','total_spend','RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']
categorical_features = ['HomePlanet','Destination','deck','side']
binary_like = ['CryoSleep','VIP','is_alone']

numeric_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
binary_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent'))])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
    ('bin', binary_transformer, binary_like)
], remainder='drop')

clf = RandomForestClassifier(random_state=42, n_estimators=200, n_jobs=1, class_weight='balanced')
pipeline = Pipeline(steps=[('fe', fe), ('pre', preprocessor), ('clf', clf)])

In [ ]:
# Train/validate split, fit, evaluate
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_val)
y_proba = pipeline.predict_proba(X_val)[:,1]
print('Validation accuracy:', accuracy_score(y_val, y_pred))
print('Validation ROC AUC:', roc_auc_score(y_val, y_proba))
print('\nClassification report:\n', classification_report(y_val, y_pred))


In [ ]:
# Save artifacts
MODEL_PATH = 'spaceship_pipeline_rf_simple.joblib'
joblib.dump(pipeline, MODEL_PATH)
print('Saved model to:', MODEL_PATH)

VAL_PRED_PATH = 'val_predictions_simple.csv'
val_out = X_val.copy()
val_out['true'] = y_val.values
val_out['pred'] = y_pred
val_out['proba'] = y_proba
val_out.to_csv(VAL_PRED_PATH, index=False)
print('Saved val predictions to:', VAL_PRED_PATH)


In [ ]:
# from sklearn.model_selection import RandomizedSearchCV
# param_dist = {
#     'clf__n_estimators': [100, 200, 400],
#     'clf__max_depth': [6, 10, 20, None],
#     'clf__min_samples_split': [2, 5, 10],
#     'clf__min_samples_leaf': [1, 2, 4]
# }
# rs = RandomizedSearchCV(pipeline, param_dist, n_iter=12, cv=5, scoring='roc_auc', random_state=42, n_jobs=1)
# rs.fit(X_train, y_train)
# print(rs.best_params_, rs.best_score_)

This markdown cell introduces the optional section for hyperparameter tuning using `RandomizedSearchCV`.

In [ ]:
# Save artifacts
MODEL_PATH = 'spaceship_pipeline_rf_simple.joblib'
joblib.dump(pipeline, MODEL_PATH)
print('Saved model to:', MODEL_PATH)

VAL_PRED_PATH = 'val_predictions_simple.csv'
val_out = X_val.copy()
val_out['true'] = y_val.values
val_out['pred'] = y_pred
val_out['proba'] = y_proba
val_out.to_csv(VAL_PRED_PATH, index=False)
print('Saved val predictions to:', VAL_PRED_PATH)

This cell saves the trained `pipeline` to a joblib file and the validation predictions (including true labels, predictions, and probabilities) to a CSV file.

In [ ]:
# Train/validate split, fit, evaluate
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_val)
y_proba = pipeline.predict_proba(X_val)[:,1]
print('Validation accuracy:', accuracy_score(y_val, y_pred))
print('Validation ROC AUC:', roc_auc_score(y_val, y_proba))
print('\nClassification report:\n', classification_report(y_val, y_pred))